# Chapter 22 — Prior Design as Architectural Feature Engineering

Reproduces:
- Figure 22.1: Architecture × prior matrix — held-out MSE for the four
  combinations of {symmetric, asymmetric} architecture and {symmetric,
  directional} prior, demonstrating the prior--architecture pairing rule.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.attention import StdAttention, SymPSDAttention, DualAttention
from tabkernels.priors import (
    SCMPrior, SCMConfig, MLPSCMPrior, ARFPrior,
    recommend_prior, KNOWN_ARCHS,
)
from tabkernels.training import ICLTrainer

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Demonstration of `recommend_prior`

Each known architecture key maps to a configured prior. The mapping
encodes the symmetric-vs-asymmetric pairing rule of the chapter.


In [ ]:
for arch in sorted(KNOWN_ARCHS):
    p = recommend_prior(architecture=arch, task='regression', seed=0)
    print(f'{arch:<10s} -> {type(p).__name__}')
print()
for arch in ['sym_psd', 'dual']:
    p = recommend_prior(architecture=arch, task='directional', seed=0)
    print(f'{arch:<10s} (directional) -> {type(p).__name__}')


## Architecture × prior empirical matrix

Train two architectures (a single-layer ``StdAttention`` and ``SymPSDAttention``
from `tabkernels.attention`) on two priors (a symmetric MLP-SCM hybrid and a
high-edge-density SCM that injects directional structure). Evaluate within-prior.

The point is not that one architecture or one prior dominates; it is that
\textbf{the right pairing matters}: symmetric architectures pair with
symmetric priors and asymmetric architectures with directional priors. The
matrix should not show wild differences within row, but the structural
pairing rule is what Chapter 22 codifies.


In [ ]:
class WrapAttn(nn.Module):
    """Wrap an AttentionBlock to expose the trainer's call signature."""
    def __init__(self, block):
        super().__init__()
        self.block = block
    def forward(self, X_q, X_ctx, y_ctx):
        return self.block(X_q, X_ctx, y_ctx)


D = 4
arch_factories = {
    'StdAttn (sym)':  lambda: WrapAttn(StdAttention(d_in=D, d_emb=24)),
    'SymPSDAttn':     lambda: WrapAttn(SymPSDAttention(d_in=D, d_emb=24)),
}

# Symmetric prior: MLP-SCM hybrid (ARF features, MLP labels).
torch.manual_seed(7)
n_corpus = 400
mode_centres = torch.tensor([[2.0, 0.0, 0.0, 0.0],
                              [0.0, 2.0, -2.0, 0.0],
                              [-2.0, -2.0, 1.0, 1.0]])
modes = torch.randint(3, (n_corpus,))
seed_corpus = mode_centres[modes] + 0.5 * torch.randn(n_corpus, D)
arf = ARFPrior(corpus=seed_corpus, max_leaves=24)
sym_prior = MLPSCMPrior(feature_prior=arf, hidden=12, depth=2, noise_scale=0.1)

# Directional/structural prior: high-edge SCM with MLP equations.
asym_prior = SCMPrior(SCMConfig(structural='mlp', edge_prob=0.7, noise_scale=0.3))

priors = {'symmetric (MLP-SCM)': sym_prior, 'directional (SCM-MLP)': asym_prior}


def evaluate(model, prior, n_tasks=20, n_ctx=48, n_q=24):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for k in range(n_tasks):
            X_c, y_c, X_q, y_q = prior.sample_episode(n_ctx, n_q, D, seed=98765 + k)
            yhat = model(X_q, X_c, y_c)
            v = y_q.var().clamp_min(1e-3)
            total += (((yhat - y_q) ** 2).mean() / v).item()
    return total / n_tasks


# Train every (arch, prior) pair, log within-prior MSE.
results = {}
for arch_name, factory in arch_factories.items():
    for prior_name, prior in priors.items():
        torch.manual_seed(42)
        model = factory()
        trainer = ICLTrainer(prior=prior, model=model, n_steps=300,
                             n_ctx=48, n_query=24, d=D, lr=3e-3,
                             warmup_steps=20, eval_every=0, seed=1)
        trainer.train()
        mse = evaluate(model, prior)
        results[(arch_name, prior_name)] = mse
        print(f'{arch_name:<14s} on {prior_name:<22s} : {mse:.3f}')


In [ ]:
arch_names = list(arch_factories)
prior_names = list(priors)
mat = np.array([[results[(a, p)] for p in prior_names] for a in arch_names])

fig, ax = plt.subplots(1, 1, figsize=(6.0, 4.5))
im = ax.imshow(mat, cmap='viridis', aspect='auto')
for i in range(len(arch_names)):
    for j in range(len(prior_names)):
        ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center',
                color='white' if mat[i, j] > mat.mean() else 'black', fontsize=11)
ax.set_xticks(range(len(prior_names))); ax.set_yticks(range(len(arch_names)))
ax.set_xticklabels(prior_names, rotation=10); ax.set_yticklabels(arch_names)
ax.set_xlabel('prior'); ax.set_ylabel('architecture')
ax.set_title('Figure 22.1: Architecture x prior within-prior MSE\n(lower is better; 20 held-out tasks)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_22_01_arch_prior_matrix.pdf', bbox_inches='tight')
plt.show()
